# 01b — The same detection with Presidio

**Why this notebook.** Presidio is an open-source framework for detecting and anonymising personal data.
A team that tries it with its default settings on multilingual travel text can get disappointing
results: names missed, airports taken for people, ordinary words taken for booking references. This
notebook reproduces that, explains the cause setting by setting, then shows Presidio configured
properly, with Azure Language plugged in as its detector for generic personal data.

**What Presidio is today.** It started at Microsoft and is now a community project under the *Data
Privacy Stack* organisation, MIT licence, with no warranty or SLA
[Documented](https://github.com/data-privacy-stack/presidio/blob/main/docs/project_transition.md).

**What it adds to Azure Language** (it does not replace it):
- a framework to declare **your own types** (pattern + context words) that runs inside the application,
  with no container to host;
- one call that **combines several detectors** (Azure Language, patterns, other models) and can explain
  each decision;
- in its separate *anonymizer* package: masking, hashing, encryption/decryption (not used here).

Its modules for JSON (alpha) and images (beta) are not shown; notebook 01 covers those needs.

**Sections.** A. Default settings · B. Why: five default settings · C. Presidio configured ·
D. A trap found on the way: context words in Hindi · E. Takeaways

In [ ]:
import logging
import warnings
from importlib.metadata import version
from IPython.display import HTML, Markdown, display

import pii_lab as L
import presidio_lab as P

warnings.filterwarnings("ignore")
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)
print("presidio-analyzer", version("presidio_analyzer"), "| spaCy models:", P.SPACY_MODELS)
display(HTML("Colour of a detection = who found it: " + L.legend(("presidio", "azure-language"))))
samples = L.TEXT_SAMPLES + L.TRAP_SAMPLES

## A. Presidio with its default settings

The engine as created by default (English only, spaCy for names, every predefined recognizer, no
threshold), plus a booking-reference pattern written the obvious way. To keep downloads small we use
the medium English spaCy model; Presidio's default is the large one.

In [ ]:
default = P.default_engine()
try:
    default.analyze(L.TEXT_SAMPLES[1]["text"], language="fr")
except Exception as error:
    print("Analysing French with the default engine:", type(error).__name__, "-", str(error)[:120])
print("So a team without a French set-up sends every text as English.\n")

default_results = {}
for sample in samples:
    spans = P.to_spans(sample["text"], default.analyze(sample["text"], language="en"))
    default_results[sample["id"]] = spans
    display(Markdown(f'**{sample["id"]}**'))
    display(HTML(L.highlight(sample["text"], spans)))

In [ ]:
def summarise(results):
    rows, traps = [], []
    for sample in samples:
        spans = results[sample["id"]]
        if sample["id"].startswith("trap"):
            traps += [f'{s["text"]} ({s["category"]})' for s in spans]
            continue
        result = L.evaluate(sample["text"], spans, sample["gold"])
        rows.append({"sample": sample["id"], "expected": len(sample["gold"]),
                     "detected": len(sample["gold"]) - len(result["missed"]),
                     "missed": ", ".join(r["value"] for r in result["missed"]),
                     "false or extra detections": ", ".join(f'{s["text"]} ({s["category"]})' for s in result["extra"])})
    display(Markdown(L.md_table(rows)))
    print(f"Detections in the trap sentences (should be none): {len(traps)}")
    print("  ", "; ".join(traps) or "none")

summarise(default_results)

**Reading the table.** Most expected values count as "detected", but only because the engine flags a lot:
look at the last column and at the trap sentences, which contain no personal data at all. A detector
that flags everything masks everything, and the text passed to the agent loses its meaning.

## B. Why: five default settings

Each cause below is visible in Presidio's source code
([repository](https://github.com/data-privacy-stack/presidio)) and in the run above.

1. **English only.** The default configuration loads one English model
   ([conf/default.yaml](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/conf/default.yaml)).
   Other languages need a model each, and their own context words.
2. **The same score for every name.** spaCy does not give a confidence for the names it finds, so
   Presidio gives them all **0.85**
   ([ner_model_configuration.py](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/nlp_engine/ner_model_configuration.py)).
   A threshold cannot separate a real passenger from "Charles de Gaulle".
3. **Patterns ignore upper and lower case.** `[A-Z2-9]{6}` then also matches "Please", "change",
   "Claire"
   ([pattern_recognizer.py](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/pattern_recognizer.py)).
4. **Context is read only before the value**: 5 words before, 0 after
   ([lemma_context_aware_enhancer.py](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/context_aware_enhancers/lemma_context_aware_enhancer.py)).
5. **No threshold**: everything is returned, down to a score of 0.01
   ([conf/default_analyzer.yaml](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/conf/default_analyzer.yaml)).

Evidence from the run above:

In [ ]:
names = [s for spans in default_results.values() for s in spans if s["category"] == "PERSON"]
print("Scores given to the names found by spaCy:", sorted({s["score"] for s in names}))
print("Some 'names':", [s["text"] for s in names][:10])
ordinary = [s["text"] for spans in default_results.values() for s in spans
            if s["category"] == "BookingReference" and not any(c.isdigit() for c in s["text"]) and not s["text"].isupper()]
print("Ordinary words taken for booking references:", ordinary)
print("Lowest score returned:", min(s["score"] for spans in default_results.values() for s in spans))

## C. Presidio configured

What changes, each a documented Presidio option:

| Default | Configured |
|---|---|
| English model only | One spaCy model per language: en, fr, de, es; for Hindi, spaCy has no trained pipeline, so the small multilingual model |
| spaCy finds the names (flat 0.85) | **Azure Language** finds generic personal data, with the settings of notebook 01 (places recognised and left out, role words excluded, threshold), through a small custom recognizer signed in with Entra ID. Presidio's built-in Azure recognizer uses an older client library without these settings |
| Every predefined recognizer | Only Azure Language + our travel patterns + a title pattern |
| Case-insensitive patterns | Case-sensitive, same patterns and base scores as the rules of notebook 01 |
| Context words in English, before the value only | Context words per language, 5 words before and 3 after |
| Threshold 0 | Threshold 0.5 |

In [ ]:
display(Markdown(L.show_code(P.AzureLanguageRecognizer, P.configured_engine)))

In [ ]:
LANGUAGE_SETTINGS = {"piiCategories": ["Default", "Location", "Airport", "City"],
                     "excludePiiCategories": ["PersonType"],
                     "confidenceScoreThreshold": {"default": 0.6}}
configured = P.configured_engine(LANGUAGE_SETTINGS)
languages = L.detect_languages([s["text"] for s in samples])

configured_results = {}
for sample, lang in zip(samples, languages):
    spans = P.to_spans(sample["text"], configured.analyze(sample["text"], language=lang))
    configured_results[sample["id"]] = spans
    display(Markdown(f'**{sample["id"]}** ({lang})'))
    display(HTML(L.highlight(sample["text"], spans)))

In [ ]:
summarise(configured_results)
leaked = [(s["id"], v) for s in L.TEXT_SAMPLES
          for v in L.leaks(L.mask(s["text"], configured_results[s["id"]]), s["gold"])]
print("Expected values still readable after masking:", leaked or "none")

Presidio can explain each decision. Here, why a booking reference was kept:

In [ ]:
text = "Mein Buchungscode lautet RVKTPN."
for r in configured.analyze(text, language="de", return_decision_process=True):
    e = r.analysis_explanation
    print(f'{text[r.start:r.end]}: {r.entity_type}, pattern score {e.original_score}, '
          f'context word "{e.supportive_context_word}" -> final score {round(r.score, 2)}')

## D. A trap found on the way: context words in Hindi

Presidio matches context words against the **base form** (lemma) of the neighbouring words, computed by
the spaCy model of the language
([lemma_context_aware_enhancer.py](https://github.com/data-privacy-stack/presidio/blob/main/presidio-analyzer/presidio_analyzer/context_aware_enhancers/lemma_context_aware_enhancer.py)).
The multilingual model used for Hindi has no lemmatizer [Observed]: every base form is empty, so no
context word can ever match. A booking reference made only of letters is then missed in Hindi, while
it is found in English:

In [ ]:
for text, lang in [("My PNR is RVKTPN", "en"), ("मेरा पीएनआर RVKTPN है", "hi")]:
    found = [(text[r.start:r.end], r.entity_type, round(r.score, 2)) for r in configured.analyze(text, language=lang)]
    print(f"{lang}: {text!r:28} -> {found or 'nothing found'}")
print("Base forms computed for the Hindi sentence:", configured.nlp_engine.process_text("मेरा पीएनआर RVKTPN है", "hi").lemmas)

The rules of notebook 01 look for context words in the raw text, so they do not depend on a
lemmatizer. With Presidio, the fix would be a Hindi pipeline that computes base forms, or a custom
context enhancer [Assumption: not tested here].

## E. Takeaways

- **With its default settings, Presidio produces exactly the kind of errors a multilingual team can
  meet**: airports and greetings taken for people, ordinary words taken for booking references, French
  analysed as English. These come from five default settings, not from the principle.
- **Configured, Presidio gives the same results as notebook 01** on these sentences. The detection of
  generic data comes from Azure Language in both cases; Presidio replaces our own glue code (patterns,
  context, merging) by a framework, and adds explanations.
- **Choosing between the two** is a question of ownership: our ~100 lines follow Microsoft's container
  rule format and have no extra dependency; Presidio is a ready framework with more features (anonymizer
  operators, JSON and image modules), but it needs a spaCy model per language, its Azure connector is
  outdated, its context words need a lemmatizer, and it is a community project without SLA.
- **Neither removes the need for the next layers**: reversible pseudonymisation around the LLM (02),
  minimisation per tool (03). Presidio provides a sample for reversible placeholders, not a ready
  feature [Documented](https://presidio.dataprivacystack.org/samples/python/pseudonymization/).